<a href="https://colab.research.google.com/github/lelongc/SKAG/blob/main/voice-srt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ô số 1: Cài đặt thư viện Whisper của OpenAI

Python

In [1]:

!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=3d86a8db0b41b306ee7c8f6eac4b762bbca0d263c8940a6888726481610e84e4
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


Ô số 2: Kết nối với Google Drive

Python

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Ô số 3: Lệnh "hủy diệt" cục audio 3 tiếng
Sếp copy đoạn này vào ô thứ 3:

In [5]:
# Lệnh "đo ni đóng giày" cho Kurisu Makise
!whisper "/content/drive/MyDrive/ggcl/kurisu/KurisuMakise1.mp3" \
--model large-v2 \
--language Japanese \
--output_dir "/content/drive/MyDrive/ggcl/kurisu/" \
--output_format srt \
--condition_on_previous_text False \ơ
--best_of 5 \
--beam_size 5 \
--initial_prompt "あんた、バka... バカなの？変態、助手、クリスティーナ。っ、あ…"

[00:01.000 --> 00:03.000] ちょっと、来てくれませんか?
[00:03.000 --> 00:04.000] は?
[00:04.000 --> 00:06.000] ふざけてないで、ちょっと来てください。
[00:06.000 --> 00:09.000] 何かって何ですか?人聞きの悪い。
[00:09.000 --> 00:12.000] 私は、あなたに聞きたいことがあるだけです。
[00:12.000 --> 00:14.000] だから、聞かんって?
[00:15.000 --> 00:19.000] あれ?この携帯、電源切れてる。
[00:19.000 --> 00:21.000] 誰と話していたんですか?
[00:21.000 --> 00:23.000] そう、独り言だったのね。
[00:23.000 --> 00:26.000] さっき、私に何を言おうとしたんですか?
[00:26.000 --> 00:33.800] ほんの15分くらい前会見が始まる前に私に何か言おうとしましたよね
[00:33.800 --> 00:37.680] すごく悲しそうな顔をしてまるで
[00:37.680 --> 00:43.680] 今にも泣き出しそうでそれにすごく 辛そうでしたどうして
[00:43.680 --> 00:48.480] 私前にあなたと会ったことありますかそういえば
[00:48.480 --> 00:53.680] なんで私の名前知ってるんですあの 何か
[00:53.680 --> 01:00.840] なんなのちょっと勝手に殺さないでくれますか 私ピンピンしてますんで
[01:00.840 --> 01:02.840] ん
[01:03.080 --> 01:12.680] おい己は警察に突き出されたいのか 何が真実よこの変態バカなの死ぬのえっ
[01:12.680 --> 01:19.000] ドクター 中鉢ねえちょっとあなた方音さん
[01:19.000 --> 01:21.560] 今の話、詳しく教えて欲しいんですが
[01:21.560 --> 01:23.560] え?あ、はい!
[01:24.500 --> 01:30.860] えーと、今日は私のような弱肺者の話を聞きに来てくださって、ありがとう
[01:

🛠️ Bước 1: Cài đặt công cụ cắt âm thanh
Sếp tạo một ô Code mới trên Colab và chạy lệnh này để cài thư viện xử lý âm thanh:

In [ ]:
!pip install pydub

🚀 Bước 2: Script "Auto-Cut" và tạo File List
Lưu ý: Sếp nhớ sửa đường dẫn file .mp3 và .srt cho đúng với tên file

In [ ]:
import os
import re
from pydub import AudioSegment

# --- CẤU HÌNH ĐƯỜNG DẪN (Sếp sửa ở đây) ---
audio_path = "/content/drive/MyDrive/ggcl/kurisu/KurisuMakise1.mp3"
srt_path = "/content/drive/MyDrive/ggcl/kurisu/KurisuMakise1.srt"
output_dir = "/content/drive/MyDrive/ggcl/kurisu/dataset_wav/" # Nơi chứa các file nhỏ
filelist_path = "/content/drive/MyDrive/ggcl/kurisu/kurisu_train_list.txt" # File danh sách để train

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Hàm chuyển đổi timestamp SRT (00:00:00,000) sang miliseconds
def srt_time_to_ms(time_str):
    time_str = time_str.replace(',', '.')
    h, m, s = time_str.split(':')
    return int(float(h)*3600000 + float(m)*60000 + float(s)*1000)

# Tải file audio gốc
print("Đang tải audio... Sếp đợi tí nhé!")
full_audio = AudioSegment.from_file(audio_path)

# Đọc file SRT
with open(srt_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Tách từng đoạn sub
blocks = re.split(r'\n\n', content.strip())
dataset_entries = []

print(f"Bắt đầu xẻ thịt {len(blocks)} câu thoại...")

for i, block in enumerate(blocks):
    lines = block.split('\n')
    if len(lines) >= 3:
        # Lấy thời gian
        times = re.findall(r'(\d{2}:\d{2}:\d{2},\d{3})', lines[1])
        if len(times) == 2:
            start_ms = srt_time_to_ms(times[0])
            end_ms = srt_time_to_ms(times[1])

            # Lấy nội dung chữ (gộp các dòng nếu có nhiều dòng sub)
            text = " ".join(lines[2:]).strip()

            # Cắt audio
            wav_filename = f"kurisu_{i:04d}.wav"
            wav_path = os.path.join(output_dir, wav_filename)

            # Chỉ lấy đoạn âm thanh tương ứng
            segment = full_audio[start_ms:end_ms]
            segment.export(wav_path, format="wav")

            # Ghi vào danh sách (Định dạng GPT-SoVITS: path|language|text)
            # Language mặc định là JA cho tiếng Nhật
            dataset_entries.append(f"{wav_path}|ja|{text}")

# Ghi file list
with open(filelist_path, 'w', encoding='utf-8') as f:
    for entry in dataset_entries:
        f.write(entry + '\n')

print(f"✅ Xong rồi sếp ơi! Đã cắt được {len(dataset_entries)} file tại: {output_dir}")
print(f"✅ File danh sách train nằm tại: {filelist_path}")